In [0]:
-- ============================================
-- PASO 1: Crear tabla de staging con detección de cambios
-- ============================================
-- Usamos tabla persistente porque:
-- - Las vistas temporales no soportan parámetros en IDENTIFIER()
-- - Las tablas temporales no soportan CREATE ... AS en este entorno

CREATE OR REPLACE TABLE IDENTIFIER(:catalogo_destino || '.dimensiones_comunes_' || :alumno || '.staging_funcionalidades_premium') AS
SELECT 
    f.id_funcionalidad,
    f.nombre_funcionalidad,
    f.descripcion,
    f.precio_base,
    f.fecha_creacion,
    f.fecha_actualizacion,
    CASE 
        WHEN d.id_funcionalidad IS NULL THEN 'NUEVO'
        WHEN f.nombre_funcionalidad != d.nombre_funcionalidad 
             OR f.descripcion != d.descripcion 
             OR f.precio_base != d.precio_base THEN 'MODIFICADO'
        ELSE 'SIN_CAMBIO'
    END as accion
FROM IDENTIFIER(:catalogo_origen || '.core_negocio_' || :alumno || '.funcionalidades_premium') f
LEFT JOIN (
    SELECT * 
    FROM IDENTIFIER(:catalogo_destino || '.dimensiones_comunes_' || :alumno || '.dim_funcionalidades_premium')
    WHERE es_actual = TRUE
) d ON f.id_funcionalidad = d.id_funcionalidad;

In [0]:
-- ============================================
-- PASO 2: Expirar registros que han cambiado
-- ============================================
-- IMPORTANTE: Usamos DATE(s.fecha_actualizacion) para valido_hasta
-- en vez de CURRENT_DATE() para que la operación sea IDEMPOTENTE.
-- Si usás CURRENT_DATE(), cada ejecución cambiaría valido_hasta
-- incluso con los mismos datos de entrada.

MERGE INTO IDENTIFIER(:catalogo_destino || '.dimensiones_comunes_' || :alumno || '.dim_funcionalidades_premium') AS target
USING IDENTIFIER(:catalogo_destino || '.dimensiones_comunes_' || :alumno || '.staging_funcionalidades_premium') AS s
ON target.id_funcionalidad = s.id_funcionalidad 
   AND target.es_actual = TRUE
   AND s.accion = 'MODIFICADO'
WHEN MATCHED THEN
    UPDATE SET
        target.valido_hasta = DATE(s.fecha_actualizacion) - INTERVAL 1 DAY,
        target.es_actual = FALSE;

In [0]:
-- ============================================
-- PASO 3: Insertar nuevos registros (nuevos + modificados)
-- ============================================
-- NOTA: id_dim_funcionalidad se genera automáticamente (IDENTITY)
-- Usamos DATE(fecha_actualizacion) como valido_desde para idempotencia
INSERT INTO IDENTIFIER(:catalogo_destino || '.dimensiones_comunes_' || :alumno || '.dim_funcionalidades_premium')
    (id_funcionalidad, nombre_funcionalidad, descripcion, precio_base, fecha_creacion, 
     fecha_actualizacion, valido_desde, valido_hasta, es_actual)
SELECT
    s.id_funcionalidad,
    s.nombre_funcionalidad,
    s.descripcion,
    s.precio_base,
    s.fecha_creacion,
    s.fecha_actualizacion,
    DATE(s.fecha_actualizacion) as valido_desde,
    DATE '9999-12-31' as valido_hasta,
    TRUE as es_actual
FROM IDENTIFIER(:catalogo_destino || '.dimensiones_comunes_' || :alumno || '.staging_funcionalidades_premium') s
WHERE s.accion IN ('NUEVO', 'MODIFICADO');

In [0]:
-- Verificar la cantidad de registros en dim_funcionalidades_premium
SELECT
    'dim_funcionalidades_premium' as tabla,
    COUNT(*) as total_registros,
    COUNT(DISTINCT id_funcionalidad) as funcionalidades_unicas,
    COUNT(CASE WHEN es_actual = TRUE THEN 1 END) as registros_actuales,
    COUNT(CASE WHEN es_actual = FALSE THEN 1 END) as registros_historicos,
    AVG(precio_base) as precio_promedio,
    MIN(precio_base) as precio_minimo,
    MAX(precio_base) as precio_maximo
FROM IDENTIFIER(:catalogo_destino || '.dimensiones_comunes_' || :alumno || '.dim_funcionalidades_premium');

In [0]:
-- Mostrar una muestra de los datos de dim_funcionalidades_premium
-- Incluye tanto registros actuales como históricos
SELECT 
    id_dim_funcionalidad,
    id_funcionalidad,
    nombre_funcionalidad,
    descripcion,
    precio_base,
    valido_desde,
    valido_hasta,
    es_actual,
    fecha_actualizacion
FROM IDENTIFIER(:catalogo_destino || '.dimensiones_comunes_' || :alumno || '.dim_funcionalidades_premium')
ORDER BY id_funcionalidad, valido_desde DESC